# RDD Fundamentals, Lineage, Transformations, Actions, and Fault Tolerance

## Learning Objectives

- Understand what an RDD is
- Create RDDs in different ways
- Understand partitions
- Learn narrow and wide transformations
- Understand actions
- Learn lineage and DAG
- Understand Spark fault tolerance
- Learn persistence and caching
- Explore Pair RDD operations
- Observe Spark UI behaviour

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.appName)
print(sc.uiWebUrl)

# 1. What is an RDD?

RDD stands for **Resilient Distributed Dataset**.

- **Resilient** → Can recover from failures using lineage.
- **Distributed** → Data is divided into partitions across executors.
- **Dataset** → Immutable collection of records.

```text
RDD
│
├── Immutable
├── Partitioned
├── Distributed
├── Fault Tolerant
└── Lazy Evaluated
```

# 2. Ways to Create an RDD

- `parallelize()`
- `textFile()`
- From existing DataFrame
- From another RDD transformation

In [ ]:
numbers = sc.parallelize(range(1,21),4)

print("Partitions:", numbers.getNumPartitions())
print(numbers.collect())

In [ ]:
print("Partition Contents")

for idx, values in enumerate(numbers.glom().collect()):
    print(f"Partition {idx}: {values}")

# 3. Immutable Nature

RDDs never change.

Every transformation creates a **new RDD**.

```text
RDD1
 │
map()
 │
 ▼
RDD2
 │
filter()
 │
 ▼
RDD3
```

In [ ]:
rdd1 = sc.parallelize([1,2,3,4,5])

rdd2 = rdd1.map(lambda x:x*10)

print(rdd1.collect())
print(rdd2.collect())

# 4. Transformations

Transformations are **lazy**.

Examples:

- map
- filter
- flatMap
- distinct
- union
- sample
- repartition
- coalesce

In [ ]:
mapped = numbers.map(lambda x:x*2)
filtered = mapped.filter(lambda x:x>20)

print("No Spark job until an action is called.")

In [ ]:
filtered.collect()

# 5. Actions

Actions trigger execution.

Examples:

- collect
- count
- first
- take
- reduce
- foreach
- saveAsTextFile

In [ ]:
print(numbers.count())
print(numbers.first())
print(numbers.take(5))
print(numbers.reduce(lambda x,y:x+y))

# 6. Narrow vs Wide Transformations

## Narrow

No data shuffle.

Examples:

- map
- filter
- flatMap

## Wide

Require shuffle.

Examples:

- groupByKey
- reduceByKey
- join
- distinct
- repartition

```text
map
 │
 ▼
Same Partition

groupByKey
 │
 ▼
Shuffle
 │
 ▼
New Stage
```

In [ ]:
words = sc.parallelize(
    ["spark","hadoop","spark","python","spark","python"],
    2
)

pairs = words.map(lambda x:(x,1))

result = pairs.reduceByKey(lambda a,b:a+b)

print(result.collect())

Observe Spark UI after running `reduceByKey()` and identify shuffle stages.

# 7. Pair RDD Operations

Common operations:

- reduceByKey
- groupByKey
- sortByKey
- combineByKey
- aggregateByKey
- join
- leftOuterJoin
- cogroup

In [ ]:
sales = sc.parallelize([
    ("North",100),
    ("North",150),
    ("South",200),
    ("East",300)
])

print(
    sales.reduceByKey(lambda a,b:a+b).collect()
)

# 8. Lineage

RDDs remember how they were created.

```text
RDD1
 │
map
 │
RDD2
 │
filter
 │
RDD3
```

Spark rebuilds lost partitions using lineage.

In [ ]:
lineage = (
    sc.parallelize(range(10))
    .map(lambda x:x*2)
    .filter(lambda x:x>5)
)

print(lineage.toDebugString())

# 9. DAG

Spark converts lineage into a Directed Acyclic Graph.

```text
RDD
 │
map
 │
filter
 │
reduceByKey
 │
Action
```

The DAGScheduler converts this DAG into stages.

# 10. Fault Tolerance

RDDs do not replicate every transformation.

Instead Spark stores **lineage**.

If Partition 2 is lost:

```text
RDD Lineage
     │
Recompute Partition 2
```

Only the missing partition is recomputed.

# 11. Persistence and Cache

Without cache:

Action 1
→ Full computation

Action 2
→ Full computation again

With cache:

Action 1
→ Compute and store

Action 2
→ Read cached partitions

In [ ]:
cached = numbers.map(lambda x:x*100).cache()

print(cached.count())
print(cached.collect())

cached.unpersist()

Check the **Storage** tab in Spark UI while the RDD is cached.

# 12. Repartition vs Coalesce (RDD)

`repartition()`

- Can increase or decrease partitions
- Causes shuffle

`coalesce()`

- Usually decreases partitions
- Avoids full shuffle where possible

In [ ]:
print("Original:", numbers.getNumPartitions())

rep = numbers.repartition(8)
print("Repartition:", rep.getNumPartitions())

coal = rep.coalesce(2)
print("Coalesce:", coal.getNumPartitions())

# 13. Spark UI Exercise

Run the following:

- map
- filter
- reduceByKey
- cache

Open:

- Jobs
- Stages
- Storage
- Executors

Observe:

- Number of stages
- Shuffle read
- Shuffle write
- Cached RDD

# 14. Interview Questions

1. What is an RDD?
2. Why is RDD immutable?
3. What is lineage?
4. What is fault tolerance?
5. Difference between map and flatMap?
6. Difference between reduceByKey and groupByKey?
7. What is a narrow transformation?
8. What is a wide transformation?
9. Why does reduceByKey perform better than groupByKey?
10. Difference between repartition and coalesce?

# 15. Key Points

```text
1. RDD = Resilient Distributed Dataset
2. RDDs are immutable.
3. Transformations are lazy.
4. Actions trigger execution.
5. Lineage provides fault tolerance.
6. Wide transformations create shuffle.
7. Narrow transformations avoid shuffle.
8. Cache stores computed partitions.
9. repartition() performs shuffle.
10. coalesce() minimizes shuffle when reducing partitions.
```

In [ ]:
# Stop Spark after completing all experiments.
# spark.stop()